Compute robustness checks for repeated runs, length controls and alternative judging.


In [1]:
import os, sys, re, json, glob
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

def _build_data_tools():
    import json
    import os
    import re
    import unicodedata
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    GOLD = str(_repo)
    PAPER = str(_repo / 'results/clinical_evaluation')
    HERE = str(_repo / 'analyses/clinical_evaluation')
    ALPHA = GOLD
    RESEARCH = GOLD
    TABLES = path_join(PAPER, 'tables')
    FIGURES = path_join(PAPER, 'figures')
    os.makedirs(TABLES, exist_ok=True)
    os.makedirs(FIGURES, exist_ok=True)
    DATASET = str(_repo / 'data' / 'limfmed_dataset_gold.csv')
    PBASE = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    PBASE_NON = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_truncated'
    ORG_DS = 'experiments/clinical_protocols/delphi_panels/heterogeneous_deepseek_organizer_complete'
    ORG_QWEN = 'experiments/clinical_protocols/delphi_panels/heterogeneous_qwen_organizer_complete'
    ORG_GEM = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gemini_external_organizer'
    PWEAK = 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro'
    PMIXED = 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm'
    PSTRONG = 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'
    HOMOG = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'deepseekv4': 'experiments/homogeneous_panels/three_deepseekv4_members_complete', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    FIX_PBASE = 'experiments/fixed_length_controls/delphi_panels/internal_panel_variants/standard_protocol/heterogeneous_gpt120_organizer'
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    MODEL_NAME = {'gpt': 'gpt-oss-120b', 'qwen': 'qwen3-235b-a22b', 'deepseekv4': 'deepseek-v4-flash', 'deepseekv4pro': 'deepseek-v4-pro', 'gemini': 'gemini-3.7-flash', 'glm': 'glm-5.2', 'kimi': 'kimi'}
    PANELS = {'P-base': (PBASE, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'gpt', True), 'P-base/org-ds': (ORG_DS, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'deepseekv4', True), 'P-base/org-qwen': (ORG_QWEN, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'qwen', True), 'P-base/org-gemini': (ORG_GEM, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'gemini', False), 'P-weak': (PWEAK, 'delphi_pweak_{d}_judge.csv', ['qwen', 'gemini', 'deepseekv4pro'], 'qwen', True), 'P-mixed': (PMIXED, 'delphi_pmixed_{d}_judge.csv', ['deepseekv4', 'gemini', 'glm'], 'deepseekv4', True), 'P-strong': (PSTRONG, 'delphi_pstrong_{d}_judge.csv', ['deepseekv4', 'glm', 'gpt'], 'deepseekv4', True), 'homog gpt': (HOMOG['gpt'], 'delphi_homog_gpt_{d}_judge.csv', ['gpt'] * 3, 'gpt', True), 'homog qwen': (HOMOG['qwen'], 'delphi_homog_qwen_{d}_judge.csv', ['qwen'] * 3, 'qwen', True), 'homog deepseekv4': (HOMOG['deepseekv4'], 'delphi_homog_deepseekv4_{d}_judge.csv', ['deepseekv4'] * 3, 'deepseekv4', True), 'homog gemini': (HOMOG['gemini'], 'delphi_homog_gemini_{d}_judge.csv', ['gemini'] * 3, 'gemini', True)}
    HETERO = ['P-base', 'P-weak', 'P-mixed', 'P-strong']
    HOMOGS = ['homog gpt', 'homog qwen', 'homog deepseekv4', 'homog gemini']
    ORGANIZER_ARMS = ['P-base', 'P-base/org-ds', 'P-base/org-qwen', 'P-base/org-gemini']
    RAW_RUN = {'P-base': PBASE + '/delphi_noroles.csv', 'P-base/org-ds': ORG_DS + '/delphi_noroles.csv', 'P-base/org-qwen': ORG_QWEN + '/delphi_noroles.csv', 'P-base/org-gemini': ORG_GEM + '/delphi_noroles.csv', 'P-weak': PWEAK + '/delphi_pweak_noroles.csv', 'P-mixed': PMIXED + '/delphi_pmixed_noroles.csv', 'P-strong': PSTRONG + '/delphi_pstrong_noroles.csv', 'homog gpt': HOMOG['gpt'] + '/delphi_homog_gpt_noroles.csv', 'homog qwen': HOMOG['qwen'] + '/delphi_homog_qwen_noroles.csv', 'homog deepseekv4': HOMOG['deepseekv4'] + '/delphi_homog_deepseekv4_noroles.csv', 'homog gemini': HOMOG['gemini'] + '/delphi_homog_gemini_noroles.csv'}
    ROUNDS_EVAL = {'P-base': (PBASE, 'delphi_base_noroles_protocol_r{r}_judge.csv'), 'P-weak': (PWEAK, 'delphi_pweak_noroles_protocol_r{r}_judge.csv'), 'P-mixed': (PMIXED, 'delphi_pmixed_noroles_protocol_r{r}_judge.csv'), 'P-strong': (PSTRONG, 'delphi_pstrong_noroles_protocol_r{r}_judge.csv'), 'homog gpt': (HOMOG['gpt'], 'delphi_homog_gpt_noroles_protocol_r{r}_judge.csv'), 'homog gemini': (HOMOG['gemini'], 'delphi_homog_gemini_noroles_protocol_r{r}_judge.csv'), 'homog deepseekv4': (HOMOG['deepseekv4'], 'delphi_homog_deepseekv4_noroles_protocol_r{r}_judge.csv')}

    def baseline_path(model, design='noroles', variant='nofix'):
        return f"experiments/{('clinical_protocols' if variant == 'nofix' else 'fixed_length_controls')}/single_model_baselines/baseline_{model}/llm_as_judge/baseline_{design}_{model}_judge.csv"

    def panel_path(name, design='noroles'):
        d, f, *_ = PANELS[name]
        return d + '/llm_as_judge/' + f.format(d=design)

    def _abs(rel):
        return rel if os.path.isabs(rel) else path_join(GOLD, rel)

    def read_csv(rel, filter_result=True):
        p = _abs(rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        d.columns = [c.lstrip('\ufeff') for c in d.columns]
        if filter_result and 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        return d

    def read_judge(rel):
        d = read_csv(rel)
        if d is None or 'Judge_score' not in d.columns:
            return None
        d = d.dropna(subset=['Judge_score', 'Vignette']).drop_duplicates('Vignette', keep='first')
        d['Judge_score'] = pd.to_numeric(d['Judge_score'], errors='coerce')
        return d.set_index('Vignette')

    def load_main_arms(design='noroles', include_usc=False):
        arms = {}
        for m in SINGLES:
            d = read_judge(baseline_path(m, design))
            if d is not None:
                arms[m] = d
        for name in PANELS:
            d = read_judge(panel_path(name, design))
            if d is not None:
                arms[name] = d
        if include_usc and design == 'noroles':
            for k in (3, 5, 7):
                d = read_judge(f'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc{k}_judge.csv')
                if d is not None:
                    arms[f'USC@{k} gpt'] = d
        return arms

    def score(arms, name):
        return arms[name]['Judge_score'].astype(float)

    def dataset():
        d = normalize_newlines(pd.read_csv(DATASET, sep=None, engine='python'))
        return d

    def protocol_text(d):
        for c in ('winner_protocol', 'baseline_protocol_anonymous', 'baseline_protocol'):
            if c in d.columns:
                return d[c].astype(str)
        raise KeyError('no protocol column')

    def strip_author(text):
        if not isinstance(text, str):
            return ''
        return re.split('(?im)^\\s*Protocol Author\\s*:', text, maxsplit=1)[0].rstrip()

    def n_words(text):
        return len(re.findall('\\S+', strip_author(text))) if isinstance(text, str) else np.nan

    def items(d):
        rows = []
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                rows.append((v, int(it['id']), int(it.get('points', 0) or 0), ax, bool(it.get('met'))))
        return pd.DataFrame(rows, columns=['Vignette', 'cid', 'points', 'axis', 'met'])

    def item_matrix(arms, names, positive_only=True):
        frames = []
        for n in names:
            it = items(arms[n]).set_index(['Vignette', 'cid'])
            frames.append(it['met'].astype(int).rename(n))
            meta = it[['points', 'axis']]
        m = pd.concat(frames, axis=1, join='inner')
        m = m.join(meta, how='left')
        if positive_only:
            m = m[m['points'] > 0]
        return m
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        x = x.strip()
        if x in GT:
            return GT[x]
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in (('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')):
            if k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        return norm_label(m.group(1)) if m else None

    def predicted_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        return protocol_text(d).map(parse_triage)

    def true_triage(d):
        return d['Type'].map(GT)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def paired(a, b):
        ix = a.dropna().index.intersection(b.dropna().index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        dd = x - y
        p = stats.wilcoxon(x, y).pvalue if (dd != 0).any() else np.nan
        return {'n': len(ix), 'delta': float(dd.mean()), 'share_gt': float((dd > 0).mean()), 'p_wilcoxon': float(p)}

    def boot_ci(x, B=10000, seed=20260922, alpha=0.05, stat=np.mean):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        idx = rng.integers(0, n, (B, n))
        s = stat(x[idx], axis=1)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_two_level(x, cluster, B=10000, seed=20260922, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c')]
        G = len(groups)
        out = np.empty(B)
        for b in range(B):
            pick = rng.integers(0, G, G)
            vals = []
            for i in pick:
                g = groups[i]
                vals.append(g[rng.integers(0, len(g), len(g))])
            out[b] = np.concatenate(vals).mean()
        return (float(np.quantile(out, alpha / 2)), float(np.quantile(out, 1 - alpha / 2)))

    def holm(p):
        p = np.asarray(p, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def mcnemar_exact(a, b):
        ix = a.dropna().index.intersection(b.dropna().index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n10, n01 = (int((x & ~y).sum()), int((~x & y).sum()))
        p = stats.binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
        return {'n': len(ix), 'acc_a': float(x.mean()), 'acc_b': float(y.mean()), 'n10': n10, 'n01': n01, 'p': float(p)}

    def wilson(k, n, z=1.96):
        p = k / n
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def tost_paired(dvec, margin):
        d = np.asarray(dvec, float)
        n = len(d)
        m, se = (d.mean(), d.std(ddof=1) / np.sqrt(n))
        p_low = stats.t.sf((m + margin) / se, n - 1)
        p_up = stats.t.cdf((m - margin) / se, n - 1)
        return float(max(p_low, p_up))

    def fmt_p(p):
        if p is None or not np.isfinite(p):
            return 'n/a'
        if p < 0.0001:
            return f'{p:.1e}'
        return f'{p:.3f}' if p >= 0.001 else f'{p:.4f}'

    def save_table(df, name, index=True):
        path = path_join(TABLES, name)
        df.to_csv(path, sep=';', encoding='utf-8-sig', index=index)
        print(f'saved tables/{name} ({len(df)} rows)')
        return path
    return locals()
from types import SimpleNamespace
P = SimpleNamespace(**_build_data_tools())
del _build_data_tools
pd.set_option('display.width', 220)
arms = P.load_main_arms()


In [2]:
RET = {'P-base': P.PBASE + '/llm_as_judge/judge_robustness/delphi_base_noroles_retest_r{r}_judge.csv', 'gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_noroles_gpt_retest_r{r}_judge.csv', 'qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_noroles_qwen_retest_r{r}_judge.csv', 'deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_noroles_deepseekv4_retest_r{r}_judge.csv'}
rows, runsc = ([], {})
for s_, pat in RET.items():
    M = pd.concat([P.read_judge(pat.format(r=r)).Judge_score.rename(r) for r in range(1, 6)], axis=1).dropna()
    runsc[s_] = M
    means = M.mean()
    long = M.stack().reset_index()
    long.columns = ['case', 'run', 'y']
    fit = smf.ols('y ~ C(case) + C(run)', long).fit()
    an = __import__('statsmodels.api').api.stats.anova_lm(fit, typ=2)
    icc = None
    rows.append({'system': P.MODEL_NAME.get(s_, s_), 'cases': len(M), 'run_means_min': means.min(), 'run_means_max': means.max(), 'sd_of_run_means': means.std(ddof=1), 'F_run': an.loc['C(run)', 'F'], 'p_run': an.loc['C(run)', 'PR(>F)'], 'within_case_sd': M.std(axis=1, ddof=1).mean(), 'between_case_sd': M.mean(axis=1).std(ddof=1)})
rt = pd.DataFrame(rows)
display(rt.round(4))
P.save_table(rt, 'repeated_run_reliability.csv', index=False)
lead = []
for r in range(1, 6):
    a = runsc['P-base'][r]
    print(f'run {r}: P-base − best single ', {m: round(P.paired(a, runsc[m][r])['delta'], 3) for m in ['gpt', 'qwen', 'deepseekv4']})
    for m in ['gpt', 'qwen', 'deepseekv4']:
        t = P.paired(a, runsc[m][r])
        lead.append({'run': r, 'vs': P.MODEL_NAME[m], 'delta': t['delta'], 'p_wilcoxon': t['p_wilcoxon']})
P.save_table(pd.DataFrame(lead), 'repeated_run_leading_systems.csv', index=False)


,system,cases,run_means_min,run_means_max,sd_of_run_means,F_run,p_run,within_case_sd,between_case_sd
0,P-base,60,0.7954,0.8156,0.0073,0.4258,0.7900,0.0778,0.0929
1,gpt-oss-120b,60,0.6283,0.6545,0.0097,0.7739,0.5431,0.0756,0.1491
2,qwen3-235b-a22b,60,0.5662,0.5870,0.0108,0.8111,0.5191,0.0825,0.1331
3,deepseek-v4-flash,60,0.6469,0.6788,0.0115,1.0149,0.4004,0.0799,0.1361


saved tables/repeated_run_reliability.csv (4 rows)
run 1: P-base − best single  {'gpt': 0.166, 'qwen': 0.237, 'deepseekv4': 0.139}
run 2: P-base − best single  {'gpt': 0.167, 'qwen': 0.234, 'deepseekv4': 0.141}
run 3: P-base − best single  {'gpt': 0.187, 'qwen': 0.249, 'deepseekv4': 0.157}
run 4: P-base − best single  {'gpt': 0.16, 'qwen': 0.208, 'deepseekv4': 0.148}
run 5: P-base − best single  {'gpt': 0.151, 'qwen': 0.219, 'deepseekv4': 0.127}
saved tables/repeated_run_leading_systems.csv (15 rows)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\results\\clinical_evaluation\\tables\\repeated_run_leading_systems.csv'

In [3]:
SH = {'P-base': P.PBASE + '/llm_as_judge/judge_robustness/delphi_base_noroles_short_judge.csv', 'gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_noroles_gpt_short_judge.csv', 'qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_noroles_qwen_short_judge.csv', 'deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_noroles_deepseekv4_short_judge.csv'}
sh = {k: P.read_judge(v) for k, v in SH.items()}
print('median input chars short vs full:', sh['P-base'].case_text.astype(str).str.len().median(), arms['P-base'].case_text.astype(str).str.len().median())
rows = []
for k in SH:
    r = P.paired(sh[k].Judge_score, arms[k].Judge_score)
    rows.append({'system': P.MODEL_NAME.get(k, k), 'full': arms[k].Judge_score.mean(), 'short': sh[k].Judge_score.mean(), 'short_minus_full': r['delta'], 'p': r['p_wilcoxon']})
for m in ['gpt', 'qwen', 'deepseekv4']:
    r = P.paired(sh['P-base'].Judge_score, sh[m].Judge_score)
    rows.append({'system': f'SHORT: P-base − {P.MODEL_NAME[m]}', 'short_minus_full': r['delta'], 'p': r['p_wilcoxon']})
shr = pd.DataFrame(rows)
display(shr.round(4))
P.save_table(shr, 'shortened_response_controls.csv', index=False)


median input chars short vs full: 177.0 759.0


,system,full,short,short_minus_full,p
0,P-base,0.7899,0.7267,-0.0632,0.0
1,gpt-oss-120b,0.6258,0.5429,-0.0829,0.0
2,qwen3-235b-a22b,0.5622,0.4683,-0.0940,0.0
3,deepseek-v4-flash,0.6401,0.5319,-0.1094,0.0
4,SHORT: P-base − gpt-oss-120b,NaN,NaN,0.1838,0.0
5,SHORT: P-base − qwen3-235b-a22b,NaN,NaN,0.2585,0.0
6,SHORT: P-base − deepseek-v4-flash,NaN,NaN,0.1948,0.0


saved tables/shortened_response_controls.csv (7 rows)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\results\\clinical_evaluation\\tables\\shortened_response_controls.csv'

In [4]:
pb = P.score(arms, 'P-base')
M = pd.concat([P.score(arms, m).rename(m) for m in ['gpt', 'qwen', 'deepseekv4']], axis=1).dropna()
gap = (pb.reindex(M.index) - M.max(axis=1)).rename('gap')
meta = arms['P-base'][['Genealogy', 'Type', 'Age Group', 'Sub Clinical Specialties']].reindex(gap.index)
rows = []
for col in meta.columns:
    groups = [g.values for _, g in gap.groupby(meta[col]) if len(g) >= 5]
    kw = stats.kruskal(*groups)
    for lev, g in gap.groupby(meta[col]):
        if len(g) >= 8:
            rows.append({'factor': col, 'level': lev, 'n': len(g), 'gap': g.mean(), 'p_within': stats.wilcoxon(g).pvalue, 'KW_p_factor': kw.pvalue})
sg = pd.DataFrame(rows)
display(sg.round(4))
print('levels with n>=8:', len(sg), '| significant within level:', int((sg.p_within < 0.05).sum()))
P.save_table(sg, 'clinical_subgroup_comparisons.csv', index=False)
g = pd.concat([pb.rename('pb'), M], axis=1).join(meta)
print(g.groupby('Genealogy')[['pb', 'gpt', 'qwen', 'deepseekv4']].mean().round(3))


,factor,level,n,gap,p_within,KW_p_factor
0,Genealogy,AI,151,0.0875,0.0000,0.5548
1,Genealogy,checker,43,0.1035,0.0000,0.5548
2,Genealogy,msd,34,0.0728,0.0014,0.5548
3,Type,Requires emergent care,84,0.0845,0.0000,0.5365
4,Type,Requires non-emergent care,80,0.0996,0.0000,0.5365
5,Type,Self-care appropriate,64,0.0791,0.0000,0.5365
6,Age Group,children (<15),35,0.0971,0.0004,0.7797
7,Age Group,elderly age (>60),37,0.0747,0.0048,0.7797
8,Age Group,middle age (45-60),44,0.0916,0.0000,0.7797
9,Age Group,young age (25-44),78,0.0869,0.0000,0.7797


levels with n>=8: 24 | significant within level: 21
saved tables/clinical_subgroup_comparisons.csv (24 rows)
              pb    gpt   qwen  deepseekv4
Genealogy                                 
AI         0.797  0.647  0.569       0.651
checker    0.743  0.548  0.502       0.568
msd        0.813  0.625  0.600       0.684


In [5]:
rr = {m: P.read_judge(P.baseline_path(m, 'randomroles')) for m in ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm']}
rr['P-base'] = P.read_judge(P.panel_path('P-base', 'randomroles'))
rows = []
for k, d in rr.items():
    r = P.paired(d.Judge_score, arms[k].Judge_score)
    rows.append({'system': P.MODEL_NAME.get(k, k), 'noroles': arms[k].Judge_score.mean(), 'randomroles': d.Judge_score.mean(), 'roles_minus_noroles': r['delta'], 'p': r['p_wilcoxon']})
for m in ['gpt', 'qwen', 'deepseekv4']:
    r = P.paired(rr['P-base'].Judge_score, rr[m].Judge_score)
    rows.append({'system': f'RANDOMROLES: P-base − {P.MODEL_NAME[m]}', 'roles_minus_noroles': r['delta'], 'p': r['p_wilcoxon']})
rrt = pd.DataFrame(rows)
display(rrt.round(4))
P.save_table(rrt, 'random_role_assignment_effects.csv', index=False)


,system,noroles,randomroles,roles_minus_noroles,p
0,gpt-oss-120b,0.6258,0.6488,0.0229,0.0519
1,qwen3-235b-a22b,0.5622,0.5952,0.0329,0.0001
2,deepseek-v4-flash,0.6401,0.6614,0.0203,0.0130
3,deepseek-v4-pro,0.5601,0.5994,0.0393,0.0003
4,gemini-3.7-flash,0.4247,0.5248,0.1001,0.0000
5,glm-5.2,0.5870,0.6552,0.0682,0.0000
6,P-base,0.7899,0.8133,0.0234,0.0028
7,RANDOMROLES: P-base − gpt-oss-120b,NaN,NaN,0.1645,0.0000
8,RANDOMROLES: P-base − qwen3-235b-a22b,NaN,NaN,0.2181,0.0000
9,RANDOMROLES: P-base − deepseek-v4-flash,NaN,NaN,0.1519,0.0000


saved tables/random_role_assignment_effects.csv (10 rows)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\results\\clinical_evaluation\\tables\\random_role_assignment_effects.csv'

In [6]:
V = P.path_join(P.GOLD, 'evaluations/human_and_model_validation')
key = pd.read_csv(P.path_join(V, 'blind_protocols_60_key.csv'), sep=';', encoding='utf-8-sig')

def read_blind(path):
    d = pd.read_csv(path, sep=';', encoding='utf-8-sig')
    return d
mini = read_blind(P.path_join(V, 'gpt41_mini_judging', 'blind_protocols_60_gpt41_mini_judge.csv'))
g41 = read_blind(P.path_join(V, 'gpt41_judging', 'judge_gpt41', 'blind_protocols_60_gpt41_judge.csv'))
print(mini.columns.tolist()[:40])
src_cols = [c for c in key.columns if c.startswith('source_of_')]
print(src_cols, key[['case_id'] + src_cols].head(2))


['case_id', 'Vignette', 'case_text', 'Criterion', 'Diagnosis_vulgar', 'Type', 'Genealogy', 'Sub Clinical Specialties', 'Age Group', 'blind_slot', 'protocol_column', 'protocol_source', 'protocol', 'Judge_score', 'Judge_analysis', 'judge_earned', 'judge_penalties', 'judge_max', 'judge_details', 'judge_prompt_tokens', 'judge_completion_tokens', 'judge_total_tokens', 'judge_cached_calls', 'uni_details', 'uni_prompt_tokens', 'uni_completion_tokens', 'uni_total_tokens', 'uni_cached_calls', 'protocol_text_matches_judged_source', 'source_judge_csv']
['source_of_1', 'source_of_2', 'source_of_3', 'source_of_4']       case_id source_of_1 source_of_2 source_of_3 source_of_4
0  951a7c6bbb        qwen      delphi  deepseekv4         gpt
1  6f3d2f1324         gpt        qwen  deepseekv4      delphi


In [7]:
def scores_by_source(d):
    k = key.set_index('case_id')
    rows = []
    for _, r in d.iterrows():
        src = k.loc[r.case_id, f'source_of_{int(r.blind_slot)}']
        rows.append({'case_id': r.case_id, 'slot': int(r.blind_slot), 'source': src, 'score': float(r.Judge_score), 'details': r.judge_details})
    return pd.DataFrame(rows)
A, B = (scores_by_source(mini), scores_by_source(g41))
AB = A.merge(B, on=['case_id', 'slot', 'source'], suffixes=('_mini', '_41'))
print('protocol scores: n =', len(AB), ' Spearman =', round(stats.spearmanr(AB.score_mini, AB.score_41)[0], 3), ' Pearson =', round(np.corrcoef(AB.score_mini, AB.score_41)[0, 1], 3))
Y = AB[['score_mini', 'score_41']].to_numpy()
n, k = Y.shape
gm = Y.mean()
msr = k * ((Y.mean(1) - gm) ** 2).sum() / (n - 1)
msc = n * ((Y.mean(0) - gm) ** 2).sum() / (k - 1)
sse = ((Y - Y.mean(1, keepdims=True) - Y.mean(0, keepdims=True) + gm) ** 2).sum()
mse = sse / ((n - 1) * (k - 1))
print('ICC(2,1) =', round((msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n), 3))
bysrc = AB.groupby('source')[['score_mini', 'score_41']].mean()
display(bysrc.round(3))
sj = []
for j in ['score_mini', 'score_41']:
    W = AB.pivot(index='case_id', columns='source', values=j)
    first = (W.idxmax(axis=1) == 'delphi').mean()
    tests = {s: (round(P.paired(W['delphi'], W[s])['delta'], 3), f"{P.paired(W['delphi'], W[s])['p_wilcoxon']:.1e}") for s in ['deepseekv4', 'gpt', 'qwen']}
    print(j, 'Delphi ranked first in', round(first, 3), 'of cases; Delphi − single:', tests)
    for s in ['deepseekv4', 'gpt', 'qwen']:
        t = P.paired(W['delphi'], W[s])
        sj.append({'judge': {'score_mini': 'gpt-4.1-mini', 'score_41': 'gpt-4.1'}[j], 'vs': P.MODEL_NAME[s], 'n': len(W), 'delta': t['delta'], 'p_wilcoxon': t['p_wilcoxon'], 'delphi_first_share': first})
P.save_table(pd.DataFrame(sj), 'independent_judge_comparisons.csv', index=False)


protocol scores: n = 240  Spearman = 0.785  Pearson = 0.812
ICC(2,1) = 0.793


,score_mini,score_41
source,,
deepseekv4,0.623,0.585
delphi,0.770,0.747
gpt,0.585,0.535
qwen,0.557,0.508


score_mini Delphi ranked first in 0.7 of cases; Delphi − single: {'deepseekv4': (0.147, '1.1e-08'), 'gpt': (0.185, '2.5e-09'), 'qwen': (0.213, '3.0e-10')}
score_41 Delphi ranked first in 0.7 of cases; Delphi − single: {'deepseekv4': (0.162, '7.8e-09'), 'gpt': (0.211, '1.7e-10'), 'qwen': (0.239, '6.9e-10')}
saved tables/independent_judge_comparisons.csv (6 rows)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\results\\clinical_evaluation\\tables\\independent_judge_comparisons.csv'

In [8]:
from sklearn.metrics import cohen_kappa_score
rows = []
for _, r in AB.iterrows():
    a = {it['id']: it for it in json.loads(r.details_mini)}
    b = {it['id']: it for it in json.loads(r.details_41)}
    for i in a:
        if i in b:
            ax = next((t.split(':', 1)[1] for t in a[i].get('tags', []) if t.startswith('axis:')), None)
            rows.append({'axis': ax, 'mini': int(bool(a[i]['met'])), 'g41': int(bool(b[i]['met'])), 'case': r.case_id})
IT = pd.DataFrame(rows)
print('item verdicts:', len(IT), ' kappa overall =', round(cohen_kappa_score(IT.mini, IT.g41), 3), ' agreement =', round((IT.mini == IT.g41).mean(), 3))
kax = IT.groupby('axis').apply(lambda g: pd.Series({'n': len(g), 'kappa': cohen_kappa_score(g.mini, g.g41), 'agree': (g.mini == g.g41).mean()}))
display(kax.round(3))
rng = np.random.default_rng(20260922)
cases = IT.case.unique()
ks = []
for _ in range(2000):
    pick = rng.choice(cases, len(cases))
    s = pd.concat([IT[IT.case == c] for c in pick])
    ks.append(cohen_kappa_score(s.mini, s.g41))
print('kappa 95% CI (case bootstrap):', np.round(np.quantile(ks, [0.025, 0.975]), 3))
P.save_table(kax, 'judge_agreement_by_scoring_axis.csv')


item verdicts: 3408  kappa overall = 0.804  agreement = 0.903


,n,kappa,agree
axis,,,
accuracy,1052.0,0.879,0.940
communication_quality,240.0,0.557,0.792
completeness,936.0,0.794,0.901
reasoning,304.0,0.707,0.872
uncertainty,344.0,0.777,0.904
urgency,532.0,0.804,0.904


kappa 95% CI (case bootstrap): [0.778 0.831]
saved tables/judge_agreement_by_scoring_axis.csv (6 rows)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\results\\clinical_evaluation\\tables\\judge_agreement_by_scoring_axis.csv'

In [9]:
def sweep(folder, pat):
    out = {}
    for f in glob.glob(P.path_join(P.GOLD, folder, '*_judge.csv')):
        m = re.search(pat, os.path.basename(f))
        if m:
            out[float(m.group(1))] = P.read_judge(f)
    return dict(sorted(out.items()))
for name, folder, pat in [('temperature', 'experiments/panel_parameter_sweeps/temperature/llm_as_judge', 'temperature_([\\d.]+)_judge'), ('questionnaire items', 'experiments/panel_parameter_sweeps/questionnaire_items/llm_as_judge', 'items_(\\d+)_judge'), ('sub-questions per expert', 'experiments/panel_parameter_sweeps/questions_per_expert/llm_as_judge', 'expert_(\\d+)_judge')]:
    S = sweep(folder, pat)
    W = pd.concat([d.Judge_score.rename(k) for k, d in S.items()], axis=1)
    full = W.dropna(axis=1, thresh=55).dropna()
    fr = stats.friedmanchisquare(*[full[c] for c in full.columns])
    means = W.mean()
    lo, hi = (min(full.columns), max(full.columns))
    tost = P.tost_paired((full[hi] - full[lo]).values, 0.05)
    print(f'{name:<26} levels {len(W.columns)} (complete {full.shape[1]}): range of means {means[full.columns].min():.3f}–{means[full.columns].max():.3f}; Friedman p = {fr.pvalue:.3f}; extreme levels {lo}→{hi} TOST±0.05 p = {tost:.4f}; n per level: {W.notna().sum().min()}–{W.notna().sum().max()}')


temperature                levels 20 (complete 17): range of means 0.792–0.834; Friedman p = 0.451; extreme levels 0.1→1.7 TOST±0.05 p = 0.0026; n per level: 1–60


questionnaire items        levels 18 (complete 18): range of means 0.796–0.840; Friedman p = 0.348; extreme levels 3.0→20.0 TOST±0.05 p = 0.0227; n per level: 60–60


sub-questions per expert   levels 10 (complete 10): range of means 0.781–0.831; Friedman p = 0.327; extreme levels 3.0→12.0 TOST±0.05 p = 0.1547; n per level: 60–60
